# Census Income Classifier (Tuned KNN)

Leakage-safe preprocessing + cross-validated KNN tuning.

## 1. Imports
Load tools for preprocessing, tuning, and evaluation.

In [15]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Columns
Define target and sensitive attributes. Sensitive attributes are excluded from model training.

In [16]:
target = "income-class"
sensitive_cols = ["marital-status", "relationship", "race", "sex", "native-country"]

## 3. Load and Split
Read data, mark `?` as missing, split first, then learn preprocessing only from training data.

In [17]:
data = pd.read_csv("adult.data.csv", na_values="?", skipinitialspace=True)

feature_cols = [c for c in data.columns if c not in [target] + sensitive_cols]
X = data[feature_cols]
y = data[target]
X_sensitive = data[sensitive_cols]

X_train, X_test, y_train, y_test, _, X_sensitivefeatures_test = train_test_split(
    X, y, X_sensitive, test_size=0.20, random_state=42, stratify=y
)

## 4. Preprocessing + KNN + CV Tuning
Numeric: median impute + scale. Categorical: mode impute + one-hot. Tune on `f1_macro`.

In [ ]:
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols),
])

pipeline = Pipeline([
    ("preprocess", preprocess),
    ("knn", KNeighborsClassifier()),
])

param_grid = {
    "knn__n_neighbors": list(range(3, 42, 2)),
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2],
    "knn__leaf_size": [20, 30, 40],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train, y_train)
classifier = search.best_estimator_

y_predict = classifier.predict(X_test)

Fitting 5 folds for each of 240 candidates, totalling 1200 fits


c:\Users\brand\anaconda3\envs\sklearn\lib\site-packages\sklearn\model_selection\_search.py:1135: UserWarning: One or more of the test scores are non-finite: [       nan 0.69043427 0.69974447 0.69191269        nan 0.69646041
 0.70606692 0.70174904        nan 0.69873652 0.71117861 0.70536755
        nan 0.70062984 0.70714752 0.70617594        nan 0.70321342
 0.71033365 0.70850828        nan 0.70210754 0.71153337 0.70982039
        nan 0.70322476 0.70966822 0.70934974        nan 0.70359704
 0.71175415 0.70983564        nan 0.70322216 0.70894845 0.70787096
        nan 0.70351214 0.70828854 0.7091681         nan 0.7033093
 0.70718316 0.70678295        nan 0.70141161 0.70702333 0.70786639
        nan 0.70138879 0.70703651 0.70737064        nan 0.70020977
 0.70457132 0.70834684        nan 0.70059557 0.7026587  0.70704864
        nan 0.70084538 0.70242223 0.70590536        nan 0.69996339
 0.702225   0.70513337        nan 0.69895925 0.70224624 0.70454475
        nan 0.69785855 0.70149977 0.7038

## 5. Results
Report best CV settings and test performance.

In [19]:
print("Best params:", search.best_params_)
print("Best CV f1_macro:", round(search.best_score_, 4))
print(confusion_matrix(y_test, y_predict))
print(classification_report(y_test, y_predict))

Best params: {'knn__leaf_size': 20, 'knn__n_neighbors': 17, 'knn__p': 2, 'knn__weights': 'uniform'}
Best CV f1_macro: 0.7118
[[4615  330]
 [ 860  708]]
              precision    recall  f1-score   support

       <=50K       0.84      0.93      0.89      4945
        >50K       0.68      0.45      0.54      1568

    accuracy                           0.82      6513
   macro avg       0.76      0.69      0.71      6513
weighted avg       0.80      0.82      0.80      6513

